# 12 — Evaluate the frozen dashboard model separately

**Optional research, not a pipeline stage.** This notebook evaluates the *existing published classifier* without fitting a model, changing its cutoff, updating the dashboard, or touching existing registrations. It writes only under the active run’s `research/12_frozen_model_evaluation/`.

**Three actions:** `plan` checks readiness without opening new price files or writing a protocol; `freeze` records an immutable evaluation design before a future outcome period; `evaluate` opens local source files only after the complete period closes. Start with `plan`.

A `future_holdout` protocol must be frozen before its outcome period starts. Choose dates with your supervisor and disclose any previous exposure; this code cannot independently audit your viewing history. **April–June 2026 is previously exposed** and must use `previously_exposed`, not an untouched-test label. Existing reserved months are blocked, including months needed for lag history. No late November registration is created or modified.

This is an offline frozen-model evaluation. It does **not** prove forecasts were issued in real time. Timely prediction archiving and source availability need separate evidence. Do not tune, switch periods or select cutoffs after viewing these results.

In [1]:
import ast
import hashlib
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

REPOSITORY = Path.cwd().resolve()
if not (REPOSITORY / "pricecatcher_config.py").is_file():
    REPOSITORY = REPOSITORY.parent
if not (REPOSITORY / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open the FYP project or research folder before running this notebook.")

## 1. Configure the design before looking at outcomes

Keep `ACTION = "plan"` for readiness checks. Fill the outcome dates (Monday through Sunday) and an honest exposure note before using `freeze`. No dates are chosen automatically: different windows answer different questions, and this notebook cannot know which data you have already inspected.

`RAW_DIRECTORY` must contain official local monthly files named `pricecatcher_YYYY-MM.parquet`, with `date`, `premise_code`, `item_code`, and `price`. Files are **not downloaded**. The notebook reuses frozen lookups, item exclusions and retail rules, converts eggs by the existing policy and builds features using main 05’s exact-calendar calculation. Nine weeks of earlier context are required, including the forecast-origin week and eight history weeks. All monthly files opened are recorded as exposed, even where rows fall outside the reported evaluation period.

You may freeze before future source files exist. Obtain the official files separately when available. Do not alter the normal refresh notebook to bypass a reserved month.

In [2]:
ACTION = "plan"  # plan, freeze, or evaluate
RUN_DIRECTORY = None  # None follows PRICECATCHER_DATA_DIR or data/active_run.txt
EVALUATION_KIND = "future_holdout"  # or previously_exposed; never relabel an exposed period
OUTCOME_START = None  # e.g. a genuinely unused future Monday, "YYYY-MM-DD"
OUTCOME_END = None    # final outcome Sunday, "YYYY-MM-DD"
EXPOSURE_NOTE = ""   # Who chose this period, when, and what has already been inspected?
RAW_DIRECTORY = REPOSITORY / "data" / "evaluation_raw"
PROTOCOL_FILE = None  # evaluate: paste the protocol.json path printed by freeze

## 2. Read-only provenance and period guards

An existing deployment pin, matching comparison/study, original inputs, published model identity, Python environment and package versions are required. The notebook does not call the helper that creates deployment pins. It hashes protected artifacts before and after evaluation. A changed manifest or environment stops execution rather than rewriting provenance.

In [3]:
def file_hash(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()


def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def inside(root, relative):
    path = (Path(root) / relative).resolve()
    if not path.is_relative_to(Path(root).resolve()):
        raise ValueError('A saved path escapes its declared root.')
    return path


def read_json(path):
    return json.loads(Path(path).read_text())


def calculations():
    """Load calculations only; importing project settings would create directories."""
    namespace = {'Path': Path, 'np': np, 'pd': pd, 'hashlib': hashlib, 'json': json,
                 '__file__': str(REPOSITORY / 'pricecatcher_config.py'),
                 'PROJECT_ROOT': RUN, 'DATA_HOME': REPOSITORY / 'data'}
    tree = ast.parse((REPOSITORY / 'pricecatcher_config.py').read_text())
    mapping = next(node for node in tree.body if isinstance(node, ast.Assign)
                   and any(isinstance(t, ast.Name) and t.id == 'RESEARCH_NOTEBOOK_NAMES' for t in node.targets))
    wanted = {'installed_package_version', 'resolve_input', 'notebook_path'}
    nodes = [mapping] + [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name in wanted]
    exec(compile(ast.Module(body=nodes, type_ignores=[]), 'read-only path helpers', 'exec'), namespace)
    for filename, names in [('05_features_and_labels.ipynb', {'make_features'}),
                            ('12_shap_and_dashboard_inputs.ipynb', {'model_inputs'})]:
        nodes = [node for cell in read_json(REPOSITORY / filename)['cells'] if cell['cell_type'] == 'code'
                 for node in ast.parse(''.join(cell['source'])).body
                 if isinstance(node, ast.FunctionDef) and node.name in names]
        if sorted(node.name for node in nodes) != sorted(names):
            raise ValueError(f'Missing or duplicate calculation in {filename}.')
        exec(compile(ast.Module(body=nodes, type_ignores=[]), filename, 'exec'), namespace)
    return namespace


def frozen_checkpoint():
    """Require an existing pin; never call the code that creates a new deployment."""
    protected = {}
    def remember(path, expected=None):
        path = Path(path).resolve()
        actual = file_hash(path)
        if expected is not None and actual != expected:
            raise ValueError(f'Frozen input changed: {path}. Restore it; do not change hashes.')
        protected[str(path)] = actual
        return path
    pin_path = RUN / '11_deployment/deployment_selection.json'
    if not pin_path.is_file():
        raise FileNotFoundError('Restore the existing deployment pin and matching run. This notebook never creates one.')
    pin = read_json(remember(pin_path))
    study_path = remember(inside(RUN, pin['study_manifest']), pin['study_manifest_sha256'])
    comparison_path = remember(inside(RUN, pin['comparison_manifest']), pin['comparison_manifest_sha256'])
    study, comparison = read_json(study_path), read_json(comparison_path)
    selected = comparison['studies'][pin['candidate']]
    if (study.get('status') != 'completed' or comparison.get('status') != 'completed'
            or comparison['candidate_under_fold_rule'] != pin['candidate']
            or inside(RUN, selected['folder']) != study_path.parent
            or selected['manifest_sha256'] != pin['study_manifest_sha256']):
        raise ValueError('The pin, comparison and study do not identify the same completed model.')
    expected_id = canonical_hash({'study_sha256': pin['study_manifest_sha256'], 'cutoff': pin['score_cutoff']})[:16]
    if expected_id != pin['deployment_id']:
        raise ValueError('Deployment identity changed.')
    selection = read_json(remember(study_path.parent / 'selection.json'))
    if selection != study['selection'] or selection['score_cutoff'] != pin['score_cutoff']:
        raise ValueError('Saved selection/cutoff changed.')
    cutoff = float(selection['score_cutoff'])
    if not np.isfinite(cutoff) or not 0 <= cutoff <= 1 or selection['surge_threshold_pct'] != 10:
        raise ValueError('Expected the frozen score cutoff and existing 10% target.')
    for name, expected in comparison['output_sha256'].items():
        remember(inside(comparison_path.parent, name), expected)
    for name, expected in study['output_sha256'].items():
        remember(inside(study_path.parent, name), expected)
    for name, expected in study['input_sha256'].items():
        remember(inside(RUN, name), expected)
    helpers = calculations()
    for name, expected in study.get('additional_input_sha256', {}).items():
        remember(helpers['resolve_input'](name), expected)
    python_version = study.get('python_version', '3.12')
    if tuple(map(int, python_version.split('.')[:2])) != sys.version_info[:2]:
        raise ValueError(f'Use the saved Python {python_version} environment.')
    for name, expected in study['package_versions'].items():
        if helpers['installed_package_version'](name) != expected:
            raise ValueError(f'Restore the saved {name} version before evaluation.')
    settings_path = RUN / 'experiments/holdout/05_modelling/experiment_settings.json'
    relative = settings_path.relative_to(RUN).as_posix()
    remember(settings_path, study['input_sha256'][relative])
    settings = read_json(settings_path)
    if settings['feature_columns'] != selection['feature_columns']:
        raise ValueError('Frozen feature lists differ.')
    names = {'LightGBM': 'lightgbm.joblib', 'XGBoost': 'xgboost.joblib',
             'Random Forest': 'random_forest.joblib', 'Logistic Regression': 'logistic_regression.joblib'}
    model = remember(study_path.parent / names[selection['family']],
                     study['output_sha256'][names[selection['family']]])
    pointer = RUN / '08_dashboard/active_deployment.json'
    dashboard = RUN / '08_dashboard'
    if pointer.exists():
        remember(pointer)
        dashboard = inside(dashboard, read_json(pointer)['directory'])
    entry_path = remember(dashboard / 'dashboard_manifest.json')
    published = read_json(entry_path)['forecast_model']
    if (published.get('deployment_id') != pin['deployment_id'] or published['sha256'] != file_hash(model)
            or published['score_cutoff'] != cutoff):
        raise ValueError('The published dashboard model differs from the pin.')
    for name in ['refresh_manifest.json']:
        if (dashboard / name).exists():
            remember(dashboard / name)
    for path in sorted((RUN / '13_registered_future_evaluation').glob('*/registration.json')):
        remember(path)
    source_path = remember(RUN / '01_integrated/source_manifest.json', settings['preparation']['source_manifest_sha256'])
    source = read_json(source_path)
    for name, expected in source['source_files_sha256'].items():
        remember(inside(source_path.parent, name), expected)
    for path in [REPOSITORY / 'pricecatcher_config.py', REPOSITORY / '05_features_and_labels.ipynb',
                 REPOSITORY / '12_shap_and_dashboard_inputs.ipynb']:
        remember(path)
    return pin, selection, settings, model, protected, helpers


def period_design(start, end, kind, note, raw_directory, training_end):
    if start is None or end is None or not str(note).strip():
        raise ValueError('Set outcome dates and an honest exposure note before freezing a protocol.')
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if (start.tzinfo is not None or end.tzinfo is not None or start != start.normalize()
            or end != end.normalize() or start.day_of_week != 0 or end.day_of_week != 6 or start > end):
        raise ValueError('Outcome period must start on Monday and end on Sunday, using date-only values.')
    if start <= pd.Timestamp(training_end):
        raise ValueError('Outcomes must follow the selected model training period.')
    if kind not in {'future_holdout', 'previously_exposed'}:
        raise ValueError('Choose future_holdout or previously_exposed explicitly.')
    today = pd.Timestamp.now(tz='Asia/Kuala_Lumpur').tz_localize(None).normalize()
    if kind == 'future_holdout' and start <= today:
        raise ValueError('A new future-holdout protocol must be frozen before the outcome period starts. Past periods cannot regain untouched status.')
    # Eight full history weeks plus the forecast-origin week before the first outcome.
    context_start = start - pd.Timedelta(weeks=9)
    months = pd.period_range(context_start, end, freq='M').astype(str).tolist()
    return {'outcome_start': str(start.date()), 'outcome_end': str(end.date()),
            'context_start': str(context_start.date()), 'raw_months': months,
            'raw_directory': str(Path(raw_directory).expanduser().resolve()),
            'evaluation_kind': kind, 'exposure_note': str(note).strip(),
            'primary_metric': 'average_precision', 'cutoff_policy': 'frozen deployment cutoff; no selection',
            'loaded_months_policy': 'whole local monthly files are opened; treat those months as exposed afterwards',
            'timely_forecasts_archived': False}


def protect_registrations(design):
    """Check whole loaded months, including history, before opening any price file."""
    for path in sorted((RUN / '13_registered_future_evaluation').glob('*/registration.json')):
        registration = read_json(path)
        content = {k: v for k, v in registration.items() if k != 'registration_sha256'}
        if canonical_hash(content) != registration['registration_sha256']:
            raise ValueError('Existing registration changed; restore its original file.')
        reserved = pd.period_range(registration['design']['outcome_start'],
                                   registration['design']['outcome_end'], freq='M').astype(str)
        if set(design['raw_months']) & set(reserved):
            raise ValueError('The evaluation or history months overlap an existing registration. Nothing is downloaded or opened; do not bypass this guard.')


def verify_unchanged(protected):
    for name, expected in protected.items():
        if file_hash(name) != expected:
            raise ValueError(f'Protected input changed: {name}. No completed evaluation can be published.')

## 3. Build an isolated evaluation population

Raw prices are joined to the frozen lookup snapshots. Invalid prices, missing metadata, wholesale rows (when excluded by the frozen policy) and previously excluded sparse items are audited. Statistical price flags are retained, as in the published workflow. No source correction or new cleaning threshold is learned.

The label remains a supported local median rise of at least **10% in the immediately following calendar week**. Missing outcomes stay excluded with reasons. The report includes the supported population, not every shop or district; a planned week with no eligible rows blocks publication.

In [4]:
def local_daily(design, settings, protected):
    """Prepare separate evaluation data with frozen eligibility and lookups; no learning."""
    lookup = RUN / '01_integrated/lookups'
    source_path = RUN / '01_integrated/source_manifest.json'
    source = read_json(source_path)
    for name, expected in source['source_files_sha256'].items():
        path = inside(source_path.parent, name)
        if file_hash(path) != expected:
            raise ValueError('Frozen source snapshot changed.')
        protected[str(path)] = expected
    if (canonical_hash(source['source_files_sha256']) != source['source_snapshot_sha256']
            or source['source_snapshot_sha256'] != settings['preparation']['source_snapshot_sha256']):
        raise ValueError('Frozen source identities differ.')
    protected[str(source_path)] = file_hash(source_path)
    item = pd.read_parquet(lookup / 'lookup_item.parquet')
    premise = pd.read_parquet(lookup / 'lookup_premise.parquet')
    for frame, key in [(item, 'item_code'), (premise, 'premise_code')]:
        frame[key] = pd.to_numeric(frame[key], errors='raise').astype('Int64')
        if frame[key].isna().any() or frame[key].duplicated().any():
            raise ValueError('Lookups must have complete unique codes.')
    units = item.set_index('item_code').unit.astype('string').str.strip().str.lower()
    for code in [118, 119, 120, 1109, 1110, 1111]:
        if units.get(code) != ('30 biji' if code in [1109, 1110, 1111] else '10 biji'):
            raise ValueError('Frozen egg units differ from the conversion policy.')
    item = item[['item_code', 'item', 'unit', 'item_group', 'item_category']]
    premise = premise[['premise_code', 'premise', 'state', 'district', 'premise_type']]
    item_map = {int(k): v for k, v in settings['preparation']['item_map'].items()}
    text_columns = ['project_group', 'item', 'unit', 'item_group', 'item_category', 'premise', 'state', 'district', 'premise_type']
    if settings['remove_flagged_prices']:
        raise ValueError('This notebook supports the frozen retained-flags policy only.')
    frames, audit = [], []
    for month in design['raw_months']:
        path = Path(design['raw_directory']) / f'pricecatcher_{month}.parquet'
        if not path.is_file():
            raise FileNotFoundError(f'Local monthly source missing: {path}. Obtain the official source separately; this notebook never downloads it.')
        digest = file_hash(path)
        protected[str(path.resolve())] = digest
        data = pd.read_parquet(path, columns=['date', 'premise_code', 'item_code', 'price'])
        data['date'] = pd.to_datetime(data.date, errors='raise').dt.normalize()
        if data.date.isna().any() or not data.date.dt.to_period('M').eq(pd.Period(month)).all():
            raise ValueError('Monthly file has missing or out-of-month dates.')
        data = data.loc[data.date.between(design['context_start'], design['outcome_end'])].copy()
        data['item_code'] = pd.to_numeric(data.item_code, errors='raise').astype('Int64')
        data = data.loc[data.item_code.isin(item_map)].copy()
        data['premise_code'] = pd.to_numeric(data.premise_code, errors='raise').astype('Int64')
        data['price'] = pd.to_numeric(data.price, errors='coerce').astype(float)
        if data.duplicated(['date', 'premise_code', 'item_code']).any():
            raise ValueError('Duplicate source keys need review; no automatic deduplication.')
        data['project_group'] = data.item_code.map(item_map)
        data = data.merge(item, on='item_code', how='left', validate='many_to_one').merge(
            premise, on='premise_code', how='left', validate='many_to_one')
        for name in text_columns:
            data[name] = data[name].astype('string').str.strip().replace('', pd.NA)
        data.loc[data.item_code.isin([1109, 1110, 1111]), 'price'] /= 3
        data.loc[data.item_code.isin([118, 119, 120, 1109, 1110, 1111]), 'unit'] = '10 biji'
        invalid = data.price.isna() | ~np.isfinite(data.price) | data.price.le(0)
        unknown = data[text_columns + ['premise_code']].isna().any(axis=1) & ~invalid
        wholesale = (data.premise_type.eq('Borong').fillna(False) & ~invalid & ~unknown
                     if settings['preparation']['remove_borong'] else pd.Series(False, index=data.index))
        sparse = data.item_code.isin(settings['excluded_item_codes']) & ~invalid & ~unknown & ~wholesale
        keep = ~(invalid | unknown | wholesale | sparse)
        audit.append({'month': month, 'selected_context_rows': len(data), 'invalid_removed': int(invalid.sum()),
                      'unknown_metadata_removed': int(unknown.sum()), 'wholesale_removed': int(wholesale.sum()),
                      'excluded_item_rows': int(sparse.sum()), 'retained_rows': int(keep.sum()), 'raw_sha256': digest})
        frames.append(data.loc[keep])
    daily = pd.concat(frames, ignore_index=True)
    if daily.empty or daily.date.max() < pd.Timestamp(design['outcome_end']):
        raise ValueError('No eligible source observations through the outcome end; do not score a partial period.')
    if daily.duplicated(['date', 'premise_code', 'item_code']).any():
        raise ValueError('Monthly sources overlap.')
    return daily, pd.DataFrame(audit)


def evaluation_rows(daily, settings, design, make_features):
    keys = ['project_group', 'item_code', 'item', 'unit', 'state', 'district', 'premise_type']
    data = daily.copy()
    data['week_start'] = data.date - pd.to_timedelta(data.date.dt.day_of_week, unit='D')
    weekly = data.groupby(keys + ['week_start']).agg(weekly_median_price=('price', 'median'),
        weekly_std_price=('price', 'std'), record_count=('price', 'size'),
        premise_count=('premise_code', 'nunique'), observed_days=('date', 'nunique')).reset_index()
    weekly = weekly.loc[weekly.week_start.ge(pd.Timestamp(design['context_start']))
                        & (weekly.week_start + pd.Timedelta(days=6)).le(pd.Timestamp(design['outcome_end']))]
    if weekly.empty:
        raise ValueError('No complete-scope weekly observations.')
    featured = pd.concat([make_features(part) for _, part in weekly.groupby(keys, sort=False)], ignore_index=True)
    featured['next_week_start'] = featured.week_start + pd.Timedelta(days=7)
    featured['next_week_end'] = featured.week_start + pd.Timedelta(days=13)
    featured['month'] = featured.week_start.dt.month
    number = featured.week_start.dt.isocalendar().week.astype(float)
    featured['week_sin'] = np.sin(2 * np.pi * number / 52.1775)
    featured['week_cos'] = np.cos(2 * np.pi * number / 52.1775)
    featured = featured.loc[featured.next_week_start.ge(pd.Timestamp(design['outcome_start']))
                           & featured.next_week_end.le(pd.Timestamp(design['outcome_end']))].copy()
    current = featured.record_count.ge(settings['minimum_records']) & featured.premise_count.ge(settings['minimum_premises'])
    future = featured.next_week_record_count.ge(settings['minimum_records']) & featured.next_week_premise_count.ge(settings['minimum_premises'])
    history = featured.lag_1w_price.notna() & featured.history_count_4w.ge(settings['minimum_history_4w'])
    featured['eligibility'] = np.select([~current, featured.next_week_price.isna(), ~future, ~history],
        ['insufficient_current_coverage', 'missing_next_week', 'insufficient_outcome_coverage', 'insufficient_history'], default='eligible')
    audit = featured.groupby(['next_week_start', 'eligibility']).size().rename('rows').reset_index()
    frame = featured.loc[featured.eligibility.eq('eligible')].copy()
    if frame.empty:
        raise ValueError('No eligible labelled pairs; inspect source coverage without substituting other periods.')
    expected_weeks = set(pd.date_range(design['outcome_start'], design['outcome_end'], freq='W-MON'))
    if set(frame.next_week_start) != expected_weeks:
        raise ValueError('At least one planned outcome week has no supported evaluation rows. No partial-period result published.')
    if frame.duplicated(keys + ['week_start']).any():
        raise ValueError('Evaluation rows must be unique.')
    change = frame.next_week_change_pct.to_numpy(dtype=float)
    tolerance = settings['preparation'].get('label_tolerance_pct', 0.)
    frame['surge_label'] = ((change >= 10.) | np.isclose(change, 10., rtol=0, atol=tolerance)).astype(int)
    if np.isinf(frame[settings['numeric_features']].to_numpy(dtype=float)).any():
        raise ValueError('Infinite feature values found.')
    return frame, audit

## 4. Report the frozen cutoff, including weak groups

Report AP, precision, recall, F1 and confusion counts overall, by outcome week and by food group. No cutoff or parameter search occurs. Undefined precision/recall and single-class AP remain blank rather than being presented as measured zero performance. Weekly/local observations are dependent; counts alone do not establish narrow uncertainty.

In [5]:
def metrics(frame):
    y = frame.surge_label.to_numpy(dtype=int)
    alerts = frame.predicted_surge.to_numpy(dtype=bool)
    tp, fp = int((alerts & (y == 1)).sum()), int((alerts & (y == 0)).sum())
    fn, tn = int((~alerts & (y == 1)).sum()), int((~alerts & (y == 0)).sum())
    return {'rows': len(frame), 'positives': int(y.sum()), 'alerts': int(alerts.sum()),
            'average_precision': float(average_precision_score(y, frame.model_score)) if len(np.unique(y)) == 2 else np.nan,
            'precision': tp / (tp + fp) if tp + fp else np.nan,
            'recall': tp / (tp + fn) if tp + fn else np.nan,
            'f1': 2 * tp / (2 * tp + fp + fn) if y.sum() else np.nan,
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

## 5. Freeze or evaluate once

The protocol records the exact checkpoint/cutoff, feature list, calculation fingerprint, period, source location and exposure declaration. `evaluate` uses that saved protocol, not newly edited date settings. Configuration changes between `freeze` and `evaluate` do not change the locked design. Changing calculation code or protected inputs is rejected.

Completed results are verified and reused on repetition. A partial result directory is not overwritten or automatically deleted. Repeating execution is reproduction, not another independent test. The original dashboard, training inputs, active-run pointer and November registration remain outside the output location.

In [6]:
def notebook_calculation_hash():
    notebook = read_json(REPOSITORY / 'research/12_frozen_model_evaluation.ipynb')
    source = '\n'.join(''.join(cell['source']) for cell in notebook['cells']
                       if cell['cell_type'] == 'code' and 'configuration' not in cell.get('metadata', {}).get('tags', []))
    return hashlib.sha256(source.encode()).hexdigest()


def freeze_protocol(design, checkpoint):
    pin, selection, settings, model, protected, helpers = checkpoint
    protect_registrations(design)
    design = {**design, 'candidate': pin['candidate'], 'deployment_id': pin['deployment_id'],
              'model_path': str(model), 'model_sha256': file_hash(model),
              'score_cutoff': selection['score_cutoff'], 'target_threshold_pct': 10.,
              'feature_columns': settings['feature_columns'], 'calculation_sha256': notebook_calculation_hash()}
    output = inside(RUN, 'research/12_frozen_model_evaluation') / canonical_hash(design)[:16]
    if Path(design['raw_directory']).is_relative_to(output.resolve()):
        raise ValueError('Raw sources must be separate from evaluation outputs.')
    path = output / 'protocol.json'
    if path.exists():
        existing = read_json(path)
        content = {k: v for k, v in existing.items() if k != 'protocol_sha256'}
        if canonical_hash(content) != existing['protocol_sha256'] or existing['design'] != design:
            raise ValueError('Protocol changed. Preserve its original record.')
        verify_unchanged(existing['protected_sha256'])
        return path
    verify_unchanged(protected)
    output.mkdir(parents=True, exist_ok=False)
    record = {'schema_version': 1, 'design': design,
              'frozen_at': pd.Timestamp.now(tz='Asia/Kuala_Lumpur').isoformat(),
              'exposure_status': 'user-declared; code cannot audit earlier viewing or analysis outside this notebook',
              'source_authenticity': 'local files must be obtained from the official source; recorded hashes identify those files, not publisher authenticity',
              'protected_sha256': protected}
    record['protocol_sha256'] = canonical_hash(record)
    with path.open('x') as handle:
        json.dump(record, handle, indent=2)
    return path


def evaluate_protocol(path, checkpoint):
    pin, selection, settings, model, protected, helpers = checkpoint
    path = Path(path).expanduser().resolve()
    output_root = inside(RUN, 'research/12_frozen_model_evaluation')
    if not path.is_relative_to(output_root) or path.name != 'protocol.json':
        raise ValueError('Select an existing protocol created under this run research/12_frozen_model_evaluation/.')
    record = read_json(path)
    content = {k: v for k, v in record.items() if k != 'protocol_sha256'}
    if canonical_hash(content) != record['protocol_sha256']:
        raise ValueError('Protocol digest changed.')
    design = record['design']
    if (design['deployment_id'] != pin['deployment_id'] or design['model_sha256'] != file_hash(model)
            or design['score_cutoff'] != selection['score_cutoff']
            or design['feature_columns'] != settings['feature_columns']
            or design['calculation_sha256'] != notebook_calculation_hash()):
        raise ValueError('Model, cutoff, features or evaluation calculations differ from the frozen protocol.')
    today = pd.Timestamp.now(tz='Asia/Kuala_Lumpur').tz_localize(None).normalize()
    if pd.Timestamp(design['outcome_end']) >= today:
        raise ValueError('The full outcome period is not closed yet. No source prices were opened.')
    if design['evaluation_kind'] == 'future_holdout':
        frozen_day = pd.Timestamp(record['frozen_at']).tz_convert('Asia/Kuala_Lumpur').tz_localize(None).normalize()
        if frozen_day >= pd.Timestamp(design['outcome_start']):
            raise ValueError('Protocol was not frozen before the future outcome period.')
    protect_registrations(design)
    verify_unchanged(record['protected_sha256'])
    protected.update(record['protected_sha256'])
    protected[str(path)] = file_hash(path)
    result = path.parent / 'result'
    if (result / 'evaluation_manifest.json').is_file():
        saved = read_json(result / 'evaluation_manifest.json')
        if saved['protocol_sha256'] != record['protocol_sha256']:
            raise ValueError('Saved result belongs to a different protocol.')
        verify_unchanged(saved['read_sha256'])
        for name, expected in saved['output_sha256'].items():
            if file_hash(inside(result, name)) != expected:
                raise ValueError('Saved evaluation output changed; restore its original files.')
        print('Reusing completed evaluation. This is not another independent test.')
        return result
    if result.exists():
        raise ValueError('Partial evaluation output exists. Preserve and inspect it before continuing; nothing is overwritten.')
    daily, source_audit = local_daily(design, settings, protected)
    frame, coverage = evaluation_rows(daily, settings, design, helpers['make_features'])
    # Model deserialization happens only after identity, provenance and version checks.
    pipeline = joblib.load(model)
    if list(pipeline.classes_) != [0, 1]:
        raise ValueError('Expected the frozen binary 0/1 classifier.')
    scores = pipeline.predict_proba(helpers['model_inputs'](frame, settings))[:, 1]
    if not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all():
        raise ValueError('Invalid frozen-model scores.')
    frame['model_score'] = scores
    frame['predicted_surge'] = scores >= design['score_cutoff']
    overall = pd.DataFrame([metrics(frame)])
    weekly = pd.DataFrame([{'outcome_week': str(week.date()), **metrics(part)}
                           for week, part in frame.groupby('next_week_start', sort=True)])
    groups = pd.DataFrame([{'project_group': name, **metrics(part)}
                           for name, part in frame.groupby('project_group', sort=True)])
    verify_unchanged(protected)
    result.mkdir(exist_ok=False)
    columns = list(dict.fromkeys(settings['feature_columns'] + ['item', 'unit', 'week_start',
        'next_week_start', 'next_week_end', 'next_week_price', 'next_week_change_pct',
        'surge_label', 'model_score', 'predicted_surge']))
    frame[columns].to_parquet(result / 'predictions.parquet', index=False)
    for name, table in [('overall.csv', overall), ('by_week.csv', weekly), ('by_food_group.csv', groups),
                        ('eligibility.csv', coverage), ('source_audit.csv', source_audit)]:
        table.to_csv(result / name, index=False)
    verify_unchanged(protected)
    saved = {'schema_version': 1, 'status': 'completed',
             'protocol_sha256': record['protocol_sha256'], 'design': design,
             'evaluation_kind': design['evaluation_kind'],
             'evaluation_status': 'declared future holdout; earlier exposure is not independently audited' if design['evaluation_kind'] == 'future_holdout' else 'retrospective evaluation on a previously exposed period',
             'timely_forecast_archiving_demonstrated': False,
             'completed_at': pd.Timestamp.now(tz='Asia/Kuala_Lumpur').isoformat(),
             'python_version': sys.version.split()[0], 'read_sha256': protected,
             'output_sha256': {p.name: file_hash(p) for p in sorted(result.iterdir()) if p.is_file()},
             'limitations': ['sampled local medians; changing shops and sparse coverage',
                            'historical lookup interpretation is provisional',
                            'undefined metrics are blank; single-class AP is not assessed',
                            'weeks and local series are dependent; this is not an independent-row uncertainty estimate',
                            'whole source months are exposed by this execution',
                            'no economic benefit, source correctness or live availability claim'],
             'model_fitted': False, 'cutoff_selected': False, 'dashboard_changed': False,
             'existing_registration_changed': False}
    with (result / 'evaluation_manifest.json').open('x') as handle:
        json.dump(saved, handle, indent=2)
    return result

## 6. Execute the selected action

On your Mac: select the matching saved Python 3.12 environment, then Run All with `plan`. For a chosen future period, use `freeze` **before** it starts and preserve the printed protocol path. After all outcomes are available locally, set `ACTION = "evaluate"` and `PROTOCOL_FILE` to that exact path, then Run All. Do not rerun main notebooks 01–13 for this analysis.

No real model/data artifacts are present in the cloud checkout, so this notebook has not been executed on your real data. Synthetic function checks do not establish real-data results.

In [7]:
if ACTION not in {"plan", "freeze", "evaluate"}:
    raise ValueError("ACTION must be plan, freeze, or evaluate.")
if RUN_DIRECTORY is not None:
    RUN = Path(RUN_DIRECTORY).expanduser().resolve()
elif os.environ.get("PRICECATCHER_DATA_DIR"):
    RUN = Path(os.environ["PRICECATCHER_DATA_DIR"]).expanduser().resolve()
else:
    pointer = REPOSITORY / "data" / "active_run.txt"
    if not pointer.is_file():
        raise FileNotFoundError("Restore the matching data/active_run.txt and saved run, or set RUN_DIRECTORY. Nothing is created.")
    relative = Path(pointer.read_text().strip())
    if relative.is_absolute() or relative.parts[:1] != ("runs",) or ".." in relative.parts:
        raise ValueError("Invalid active-run pointer.")
    RUN = inside(REPOSITORY / "data", relative)

checkpoint = frozen_checkpoint()
pin, selection, settings, model_path, protected, helpers = checkpoint
print("Existing published model:", pin["candidate"], "| deployment:", pin["deployment_id"])
print("Frozen score cutoff:", selection["score_cutoff"], "| target: next-week median rise >= 10%")
print("Active run:", RUN)
if ACTION == "plan":
    print("Readiness/provenance check passed. No new price files opened; no protocol or results written.")
    print("Next: choose dates and exposure note, then use freeze before a future outcome period.")
elif ACTION == "freeze":
    design = period_design(OUTCOME_START, OUTCOME_END, EVALUATION_KIND, EXPOSURE_NOTE,
                           RAW_DIRECTORY, selection["training_end"])
    protocol_path = freeze_protocol(design, checkpoint)
    print("Protocol preserved:", protocol_path)
    print("No evaluation prices opened. Whole local months needed later:", design["raw_months"])
else:
    if PROTOCOL_FILE is None:
        raise ValueError("Set PROTOCOL_FILE to an existing frozen protocol.json; evaluation never creates one.")
    result = evaluate_protocol(PROTOCOL_FILE, checkpoint)
    print("Saved separate evaluation:", result)
    manifest = read_json(result / "evaluation_manifest.json")
    print("Evidence status:", manifest["evaluation_status"])
    print("No fitting, cutoff selection, dashboard updates or registration changes.")
    for filename in ["overall.csv", "by_week.csv", "by_food_group.csv", "eligibility.csv", "source_audit.csv"]:
        print(filename)
        display(pd.read_csv(result / filename))

Existing published model: Longer-history LightGBM (tuned) | deployment: 8817e80ce245dedd
Frozen score cutoff: 0.22099133987116373 | target: next-week median rise >= 10%
Active run: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240
Readiness/provenance check passed. No new price files opened; no protocol or results written.
Next: choose dates and exposure note, then use freeze before a future outcome period.


## Interpretation for the report

Name the evaluated checkpoint, period, outcome coverage and frozen cutoff. A future-holdout protocol supports a stronger evaluation design, but its exposure declaration is not independently certified and offline feature reconstruction does not demonstrate timely forecasts. Previously exposed periods remain retrospective. Retain results that are worse than validation, disclose sparse groups, and do not replace the deployed model based on this notebook.

Outputs: `protocol.json`, then a separate `result/` containing `predictions.parquet`, overall/weekly/group metrics, eligibility/source audits and `evaluation_manifest.json`. No scores in this notebook are produced until you explicitly evaluate real local artifacts.